# Gold Forecast Lab: from coursework to an auditable experiment

This executed notebook walks through the supplied data, causal feature alignment, generated model results and the classical time-series extension. Reusable implementation lives in `gold_forecast/`; the notebook reads the same artifacts as the dashboard.

**Target:** future recorded GLD. **Horizon:** 1, 5 or 20 observed rows. **Data ends:** May 2018. No current market predictions or trading-profit claims.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'main.csv').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from gold_forecast.data import load_data, supervised, split_origins

data = load_data(ROOT / 'main.csv')
metadata = json.loads((ROOT / 'artifacts/metadata.json').read_text())
print(f'{len(data):,} observations: {data.Date.min().date()} to {data.Date.max().date()}')
print(data.head().to_string(index=False))


2,271 observations: 2008-01-02 to 2018-05-16
      Date       GLD         SPX       USO    SLV  EUR/USD  EFFR  daily_inflation_rate
2008-01-02 84.860001 1447.160034 78.470001 15.180 1.471692  4.11              0.000103
2008-01-03 85.570000 1447.160034 78.370003 15.285 1.474491  4.25              0.000103
2008-01-04 85.129997 1411.630005 77.309998 15.167 1.475492  4.18              0.000103
2008-01-07 84.769997 1416.180054 75.500000 15.053 1.468299  4.27              0.000103
2008-01-08 86.779999 1390.189941 76.059998 15.590 1.557099  4.27              0.000103


## 1. Data quality and target meaning

No cells are missing, but dates have gaps. We do not invent prices for those gaps. GLD is the supplied proxy; the file does not establish whether it is a daily high, close or adjusted price. Macro fields have no release vintages and are used only in the separate paper reconstruction.


In [2]:
print(data.drop(columns='Date').describe().round(3).to_string())
print('Duplicate dates:', data.Date.duplicated().sum())
print('Missing cells:', data.isna().sum().sum())
print('Date-gap counts (calendar days):')
print(data.Date.diff().dt.days.value_counts().sort_index().to_string())


            GLD       SPX       USO       SLV   EUR/USD      EFFR  daily_inflation_rate
count  2271.000  2271.000  2271.000  2271.000  2271.000  2271.000                2271.0
mean    122.718  1654.897    31.846    20.083     1.284     0.457                   0.0
std      23.244   519.423    19.563     7.091     0.132     0.683                   0.0
min      70.000   676.530     7.960     8.850     1.039     0.040                  -0.0
25%     109.730  1240.880    14.375    15.570     1.170     0.110                   0.0
50%     120.580  1551.690    33.870    17.267     1.303     0.160                   0.0
75%     132.830  2073.675    37.825    22.875     1.370     0.400                   0.0
max     184.590  2872.870   117.480    47.260     1.599     4.270                   0.0
Duplicate dates: 0
Missing cells: 0
Date-gap counts (calendar days):
Date
1.0     1703
2.0       28
3.0      187
4.0      310
5.0       39
6.0        1
8.0        1
18.0       1


## 2. Make the forecast origin explicit

At row `t`, origin-time features use rows at or before `t`; the target is `GLD[t+h]`. For a one-step prediction, origin-time fields correspond to the previous row relative to the target. The training cutoff is based on target indices, and multi-step CV purges overlapping future labels.


In [3]:
horizon = 5
X, y = supervised(data, horizon)
train, test = split_origins(X.index.to_numpy(), metadata['cutoff_index'], horizon)
origin = int(test[0])
print('Feature columns:', list(X.columns))
print('Last training target:', data.Date.iloc[train[-1] + horizon].date())
print('First test origin:', data.Date.iloc[origin].date())
print('First test target:', data.Date.iloc[origin + horizon].date())
assert train[-1] + horizon < metadata['cutoff_index']
assert y.loc[origin] == data.GLD.iloc[origin + horizon]
print('Alignment assertions passed.')


Feature columns: ['GLD_at_origin', 'SPX_at_origin', 'USO_at_origin', 'SLV_at_origin', 'EUR/USD_at_origin', 'GLD_lag_1', 'GLD_lag_2', 'GLD_lag_5', 'GLD_lag_10', 'GLD_lag_20', 'GLD_return_1', 'GLD_mean_5', 'GLD_volatility_5', 'GLD_mean_20', 'GLD_volatility_20']
Last training target: 2016-04-13
First test origin: 2016-04-13
First test target: 2016-04-21
Alignment assertions passed.


## 3. Compare on identical holdout targets

The main benchmark covers 11 models, including Naive, Drift, Seasonal Naive, Ridge, decision trees, bagging, Random Forest, gradient boosting, ARIMA, SARIMA and exponential smoothing. MAE is the primary metric. Positive `skill_vs_naive` indicates improvement over an unchanged-price prediction. Model orders and hyperparameters are chosen on development folds, not on this table.

![Holdout model comparison](../artifacts/figures/leaderboard.png)


In [4]:
metrics = pd.read_csv(ROOT / 'artifacts/metrics.csv')
print(metrics[['model', 'horizon', 'mae', 'rmse', 'r2', 'skill_vs_naive']].round(4).to_string(index=False))


                model  horizon    mae   rmse      r2  skill_vs_naive
               SARIMA        1 0.7305 0.9979  0.9577          0.0041
                ARIMA        1 0.7310 0.9976  0.9577          0.0034
Exponential Smoothing        1 0.7312 0.9977  0.9577          0.0032
                Drift        1 0.7332 0.9983  0.9577          0.0005
                Naive        1 0.7336 0.9983  0.9577          0.0000
                Ridge        1 0.7355 1.0066  0.9570         -0.0026
    Gradient Boosting        1 0.8199 1.0922  0.9493         -0.1177
        Decision Tree        1 0.8926 1.1945  0.9394         -0.2168
        Random Forest        1 1.0233 1.3186  0.9262         -0.3950
              Bagging        1 1.0419 1.3546  0.9221         -0.4203
       Seasonal Naive        1 1.8549 2.3079  0.7738         -1.5286
                Drift        5 1.8618 2.3163  0.7733          0.0021
                Naive        5 1.8656 2.3171  0.7732          0.0000
       Seasonal Naive        5 1.8

## 4. Small improvements need uncertainty

The paired table compares absolute forecast losses with Naive. Negative deltas favor the model. If the block-bootstrap interval spans zero, this descriptive analysis does not establish a clear improvement. These are not simultaneous confidence intervals across all candidates.


In [5]:
paired = pd.read_csv(ROOT / 'artifacts/paired_comparisons.csv')
print(paired[(paired.horizon == 1) & paired.model.isin(['ARIMA', 'SARIMA', 'Random Forest', 'Exponential Smoothing'])].round(5).to_string(index=False))
print('Classical interval coverage:')
print(pd.read_csv(ROOT / 'artifacts/interval_coverage.csv').round(4).to_string(index=False))


                model  horizon  delta_mae_vs_naive   ci_low  ci_high
                ARIMA        1            -0.00252 -0.00627  0.00168
Exponential Smoothing        1            -0.00232 -0.00588  0.00172
        Random Forest        1             0.28972  0.18176  0.39958
               SARIMA        1            -0.00302 -0.00766  0.00201
Classical interval coverage:
                model  horizon  coverage_95  mean_width
                ARIMA        1       0.9890      6.4768
                ARIMA        5       0.9956     14.0357
                ARIMA       20       0.9977     27.9086
Exponential Smoothing        1       0.9890      6.4752
Exponential Smoothing        5       0.9956     14.0377
Exponential Smoothing       20       0.9977     27.9068
               SARIMA        1       0.9890      6.4752
               SARIMA        5       0.9956     14.0276
               SARIMA       20       0.9977     28.3438


## 5. Reconstruct the paper and repair ablation

The reconstruction retains seven origin-time variables, 12-tree bagging and both 30-/50-tree RF interpretations. This is separate from the market-only main benchmark. The corrected ablation removes each feature, refits, and compares absolute errors on a development validation block. Positive ΔMAE means removing the variable hurts. Macro timing remains unverified; p-values are exploratory under temporal dependence.


In [6]:
print(pd.read_csv(ROOT / 'artifacts/paper_reconstruction.csv')[['model', 'mae', 'mse', 'r2']].round(4).to_string(index=False))
print('Feature ablation:')
print(pd.read_csv(ROOT / 'artifacts/ablation.csv')[['removed_feature', 'delta_mae', 'block_ci_low', 'block_ci_high']].round(4).to_string(index=False))


                              model    mae     mse      r2
                      Decision Tree 1.4256  3.8020  0.8385
                 Bagging (12 trees) 1.0647  1.8639  0.9208
 Random Forest (50 trees, appendix) 1.0111  1.6642  0.9293
Random Forest (30 trees, narrative) 4.6044 32.0108 -0.3595
Feature ablation:
     removed_feature  delta_mae  block_ci_low  block_ci_high
                 GLD     2.6506        1.3526         4.0070
                 SPX     0.0130       -0.0455         0.0724
                 USO     0.0835        0.0337         0.1306
                 SLV     0.0731        0.0408         0.1119
             EUR/USD     0.0701       -0.0073         0.1524
                EFFR     0.1095        0.0518         0.1747
daily_inflation_rate     0.0986        0.0563         0.1498


## 6. Reproduce and extend

From the repository root:

```bash
python -m pip install -r requirements-dev.txt
python -m gold_forecast run
python -m pytest -q
python -m streamlit run app.py
```

The full pipeline generates these outputs from scratch. For a smaller search, use `python -m gold_forecast run --quick --output artifacts/smoke`. Hyperparameter search and model fitting are deliberately centralized in the package rather than duplicated here.

Read [methodology](../docs/METHODOLOGY.md), [document audit](../docs/DOCUMENT_AUDIT.md), [data card](../docs/DATA_CARD.md), and [interview guide](../docs/INTERVIEW_GUIDE.md). A more complex model is not automatically a better forecast; honest baselines and reproducible evidence are the contribution.
